In [1]:
import json, zipfile
from pathlib import Path
import numpy as np
import pandas as pd

from asforests.cb_computer import Callback

from experiments.problem_instance.problem_instance import ProblemInstance

from experiments.benchmark.benchmark import Benchmark
from experiments.benchmark.approaches import DatabaseWiseApproach
from experiments.benchmark._ground_truth_computer import GroundTruthComputer

import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

from tqdm import tqdm

In [2]:
class XiTermPairCollector(Callback):
    
    def __init__(self):
        super().__init__()
        self.xi_pairs_list = []
       
    def on_xi_term_pair_computation(self, finished_rounds, new_xi_term_pairs):
        if len(self.xi_pairs_list) == finished_rounds:
            self.xi_pairs_list.append(new_xi_term_pairs)

# Compute and Store Approach Errors on Different Datasets and Seeds

In [ ]:
resultfile = Path("results.csv")

df_results = None if not resultfile.exists() else pd.read_csv(resultfile)

d_num_samples_for_estimation = [10**3, 10**4, 10**5, 10**6]
d_ensemble_sequence_seed = list(range(5))
max_budget = 100

for openmlid in [1049]:
    for data_seed in range(1):
        for num_instances in [100]:
            X, y = None, None
            for num_possible_ensemble_members in [8, 16]:
                for validation_size in [8, 16, 32]:
                    pi = None

                    for max_num_samples_for_estimation in d_num_samples_for_estimation:
                        for ensemble_sequence_seed in d_ensemble_sequence_seed:

                            # check whether a result exists
                            result_exists = df_results is not None and sum(
                                (df_results["openmlid"] == openmlid) &
                                (df_results["data_seed"] == data_seed) &
                                (df_results["num_instances"] == num_instances) &
                                (df_results["num_possible_ensemble_members"] == num_possible_ensemble_members) &
                                (df_results["validation_size"] == validation_size)&
                                (df_results["max_num_samples_for_estimation"] == max_num_samples_for_estimation)&
                                (df_results["ensemble_sequence_seed"] == ensemble_sequence_seed)
                            ) >= 1
                            
                            # if no result exists perform the experiment
                            if result_exists:
                                print("Skipping since result is present")
                                continue
                            
                            else:

                                # first load the data unless it has been loaded before for the suitable circumstances
                                if X is None:
                                    X, y = fetch_openml(data_id=openmlid, return_X_y=True)
                                    X, _, y, _ = train_test_split(X, y, train_size=num_instances, random_state=0)
                                    if isinstance(X, pd.DataFrame):
                                        X = X.values
                                    if isinstance(y, pd.Series):
                                        y = y.values
                                
                                # create problem instance and compute ground truth
                                pi = ProblemInstance(
                                    data_description=(X, y),
                                    is_classification=True,
                                    data_seed=data_seed,
                                    ensemble_seed=0,
                                    training_instances_per_class=0.5,
                                    num_possible_ensemble_members=num_possible_ensemble_members,
                                    validation_size=validation_size,
                                    num_samples_allowed_for_ground_truth_approximation=10**8,
                                    n_checkpoints=np.array([1, 2, 10, 100, 1000]),
                                    t_checkpoints=np.array([1, 2, 10, 100, 1000])
                                )

                                # trigger ground truth computation
                                #print(f"Computing IID ground truth")
                                #pi._compute_exact_ground_truth_iid()
                                print(f"Computing conditional case ground truth")
                                pi._compute_exact_ground_truth_cond()

                                # get approach that contains ground truth values for the right hand side of Theorem 1
                                #agt_iid = pi.approach_for_gt_iid_case
                                agt_cond = pi.approach_for_gt_conditional_case

                                # 
                                print("Initializing benchmark")
                                benchmark = Benchmark(
                                    captured_parameters=["V[Z_nt|D_val]"],
                                    problem_instance=pi,
                                    ensemble_sequence_seed=ensemble_sequence_seed
                                )
                                a = DatabaseWiseApproach(
                                    threshold_for_number_of_samples_to_exclude_param=max_num_samples_for_estimation
                                )
                                benchmark.reset(approaches={"DB": a})
                                print("Initialization ready, starting computation.")

                                # compute error in estimation of conditional covariance terms
                                e_covs_cond_setting_on_seed = []
                                for _ in tqdm(range(max_budget)):
                                    benchmark.step()
                                    e_covs_cond_setting_on_seed.append((a.xi_covs_in_conditional_setting - agt_cond.xi_covs_in_conditional_setting).tolist())
                                new_row_as_frame = pd.Series({
                                    "openmlid": openmlid,
                                    "data_seed": data_seed,
                                    "num_instances": num_instances,
                                    "num_possible_ensemble_members": num_possible_ensemble_members,
                                    "validation_size": validation_size,
                                    "max_num_samples_for_estimation": max_num_samples_for_estimation,
                                    "ensemble_sequence_seed": ensemble_sequence_seed,
                                    "error_curve_cond": e_covs_cond_setting_on_seed
                                }).to_frame().T
                                df_results = new_row_as_frame if df_results is None else pd.concat([df_results, new_row_as_frame], axis=0)
                                df_results.to_csv(resultfile, index=False)

Computing conditional case ground truth


Initializing benchmark
Initialization ready, starting computation.


100%|██████████| 100/100 [00:07<00:00, 13.93it/s]


Computing conditional case ground truth
Initializing benchmark
Initialization ready, starting computation.


100%|██████████| 100/100 [00:05<00:00, 17.04it/s]


...

100%|██████████| 100/100 [00:05<00:00, 17.86it/s]


Computing conditional case ground truth
Initializing benchmark
Initialization ready, starting computation.


100%|██████████| 100/100 [00:04<00:00, 20.75it/s]


Skipping since result is present
Skipping since result is present
Skipping since result is present
Skipping since result is present
Skipping since result is present
Skipping since result is present
Skipping since result is present
Skipping since result is present
Skipping since result is present
Skipping since result is present
Skipping since result is present
Skipping since result is present
Skipping since result is present
Skipping since result is present
Skipping since result is present
Computing conditional case ground truth
Initializing benchmark


  0%|          | 7426/5000000 [08:00<118:34:03, 11.70it/s]

# Estimate Ground Truth

## Conditional Setting

In [ ]:
for max_number_of_recent_members_to_combine_with in [10, 20, 30, 40, 50]:
    benchmark = Benchmark(problem_instance=pi)
    a = DatabaseWiseApproach(
        max_number_of_recent_members_to_combine_with=max_number_of_recent_members_to_combine_with
    )
    benchmark.reset(approaches={"DB": a})

    e_covs_cond_setting = []
    e_covs_iid_setting = []
    for _ in tqdm(range(50)):
        benchmark.step()

        e_covs_cond_setting.append(a.xi_covs_in_conditional_setting - agt_cond.xi_covs_in_conditional_setting)
        e_covs_iid_setting.append(a.xi_covs_in_iid_setting - agt_iid.xi_covs_in_iid_setting)

    e_covs_cond_setting = np.array(e_covs_cond_setting)
    e_covs_iid_setting = np.array(e_covs_iid_setting)

    # show result
    fig, axs = plt.subplots(1, 3, figsize=(20, 4))
    ax = axs[0]
    ax.plot(e_covs_iid_setting[:, :7], label=labels_iid[:7])
    ax.axhline(0, color="black")
    ax.legend()
    ax.set_title(f"Result for {max_number_of_recent_members_to_combine_with=} in IID setting")
    ax = axs[1]
    ax.plot(e_covs_iid_setting[:, 7:], label=labels_iid[7:])
    ax.axhline(0, color="black")
    ax.legend()
    ax.set_title(f"Result for {max_number_of_recent_members_to_combine_with=} in IID setting")
    ax = axs[2]
    ax.plot(e_covs_cond_setting, label=labels_cond)
    ax.axhline(0, color="black")
    ax.legend()
    ax.set_title(f"Result for {max_number_of_recent_members_to_combine_with=} in conditional setting")

    for ax in axs:
        ax.grid()

    fig.tight_layout()
    plt.show()